# 02 - Feature engineering: Mersey River streamflow

**Goal of this notebook:**
1. Reload the cleaned discharge series (redone from the raw CSV, so this notebook can run independently of 01_eda)
2. Build lag features, rolling statistics, and seasonal indicators
3. Optionally pull NASA POWER precipitation/temperature as exogenous variables
4. Define the forecasting target and do a chronological train/test split
5. Save the engineered feature table to data/processed/ for 03_modeling.ipynb

In [1]:
# --- imports ---
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)

## Step 1 - reload and re-clean the raw pull

Redoing the (small) cleaning steps here rather than relying on 01_eda's in-memory `df` keeps this
notebook runnable on its own - a standard practice so notebooks don't silently depend on each
other's kernel state.

In [2]:
raw = pd.read_csv("../data/raw/mersey_daily_discharge_raw.csv")

df = raw.rename(columns={
    "Date": "date",
    "Value/Valeur": "discharge_cms",
})[["date", "discharge_cms"]].copy()

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

# fill the small gaps (0.3% missing, confirmed in 01_eda) rather than dropping them --
# dropping would break the daily frequency the lag/rolling features below depend on.
# forward-fill is a reasonable choice for short single-day gaps in a slow-changing signal like
# river discharge; if a gap were long (multi-week), forward-fill would be a poor choice and
# we'd want to flag/drop that stretch instead -- worth a spot-check before trusting this blindly
full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "date"
df["discharge_cms"] = df["discharge_cms"].ffill()

print(f"{len(df)} daily rows, {df['discharge_cms'].isna().sum()} still missing after fill")
df.head()

8930 daily rows, 0 still missing after fill


,discharge_cms
date,
1954-12-17,69.4
1954-12-18,69.9
1954-12-19,41.6
1954-12-20,68.2
1954-12-21,70.2


## Step 2 - lag features

Past discharge values are the strongest predictor of near-future discharge (rivers don't change
instantly) -- this mirrors the lag-feature approach that worked well in the Load Forecasting project.

In [3]:
# short lags capture day-to-day persistence, longer lags capture slower drawdown after a storm
LAG_DAYS = [1, 2, 3, 7, 14, 30]

for lag in LAG_DAYS:
    df[f"discharge_lag{lag}"] = df["discharge_cms"].shift(lag)

## Step 3 - rolling statistics

Rolling means smooth out day-to-day noise and help the model see the underlying trend; rolling std
captures how volatile flow currently is (e.g. mid-storm vs. steady baseflow).

In [4]:
for window in [7, 30]:
    # shift(1) before rolling so the window only looks at PAST days, never the current
    # day's own value -- otherwise this would leak the answer into its own feature
    df[f"discharge_roll_mean_{window}"] = df["discharge_cms"].shift(1).rolling(window).mean()
    df[f"discharge_roll_std_{window}"] = df["discharge_cms"].shift(1).rolling(window).std()

## Step 4 - seasonal indicators

Plain month numbers treat December (12) and January (1) as far apart, when seasonally they're
adjacent -- sin/cos cyclical encoding fixes that by mapping day-of-year onto a circle.

In [5]:
df["month"] = df.index.month
df["day_of_year"] = df.index.dayofyear

# cyclical encoding: converts day-of-year (1-366) into a point on a circle, so the model
# understands Dec 31 and Jan 1 are adjacent rather than 365 days apart
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

# simple season bucket, useful for grouping/plotting even though the model itself
# will mostly rely on the sin/cos features above
season_map = {12: "winter", 1: "winter", 2: "winter",
              3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer",
              9: "fall", 10: "fall", 11: "fall"}
df["season"] = df["month"].map(season_map)

## Step 5 (optional) - NASA POWER precipitation/temperature

Adds precipitation and temperature as exogenous variables, the same idea used in the SARIMAX notes
(HDD/weather normalization) from the Killam work. NASA POWER needs no API key. Note this uses the
Mersey River / Milton coordinates (44.07, -64.76) -- different from the Goldboro coordinates already
hardcoded in nasa_power_api.py for the RETScreen capstone, so don't reuse that module's constants
as-is here.

This cell can be skipped for a first pass at modeling -- it's optional, come back to it if the
lag/rolling/seasonal features alone aren't enough.

In [6]:
def fetch_nasa_power_daily(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """
    Pull daily precipitation and temperature from NASA POWER for a single point.

    lat, lon : decimal degrees
    start, end : "YYYYMMDD" (NASA POWER's date format, no dashes)
    """
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "PRECTOTCORR,T2M",  # corrected precipitation (mm/day), temp at 2m (C)
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()["properties"]["parameter"]

    weather = pd.DataFrame({
        "precip_mm": payload["PRECTOTCORR"],
        "temp_c": payload["T2M"],
    })
    weather.index = pd.to_datetime(weather.index, format="%Y%m%d")
    weather.index.name = "date"
    return weather


# Mersey River at Milton coordinates -- NASA POWER's satellite-era coverage starts 1981,
# so this WON'T overlap our 1954-1979 discharge record. Left in as a template for when
# we work with a currently-active station (see the note at the end of 01_eda about pairing
# this historical station with a live one). Skip running this cell for now.
# weather = fetch_nasa_power_daily(44.0667, -64.7650, start="19810101", end="20261231")
# weather.head()

## Step 6 - define the forecasting target

We're predicting tomorrow's discharge from today's features -- a 1-day-ahead forecast, the same
framing as the Load Forecasting project's next-period prediction.

In [7]:
df["target_discharge_next_day"] = df["discharge_cms"].shift(-1)

# drop rows that don't have a full feature set yet (the first 30 days, due to the longest
# lag/rolling window) or don't have a target (the very last day, shifted off the end)
model_df = df.dropna().copy()
print(f"{len(df)} rows before dropping incomplete rows -> {len(model_df)} rows ready for modeling")
model_df.head()

8930 rows before dropping incomplete rows -> 8899 rows ready for modeling


,discharge_cms,discharge_lag1,discharge_lag2,discharge_lag3,discharge_lag7,discharge_lag14,discharge_lag30,discharge_roll_mean_7,discharge_roll_std_7,discharge_roll_mean_30,discharge_roll_std_30,month,day_of_year,doy_sin,doy_cos,season,target_discharge_next_day
date,,,,,,,,,,,,,,,,,
1955-01-16,40.5,68.2,56.1,57.2,34.8,39.6,69.4,54.757143,10.253919,56.610000,15.173215,1,16,0.271777,0.962360,winter,39.4
1955-01-17,39.4,40.5,68.2,56.1,50.4,54.4,69.9,55.571429,8.477365,55.646667,15.250409,1,17,0.288291,0.957543,winter,55.5
1955-01-18,55.5,39.4,40.5,68.2,58.3,64.3,41.6,54.000000,10.397756,54.630000,15.284050,1,18,0.304719,0.952442,winter,54.4
1955-01-19,54.4,55.5,39.4,40.5,58.3,56.6,68.2,53.600000,10.257680,55.093333,15.084816,1,19,0.321058,0.947060,winter,57.2
1955-01-20,57.2,54.4,55.5,39.4,57.2,60.6,70.2,53.042857,10.063938,54.633333,14.880381,1,20,0.337301,0.941397,winter,60.6


## Step 7 - chronological train/test split

Time series data can't use a random split -- that would let the model "see the future" during
training. Instead we hold out the last 20% of the timeline as test data.

In [8]:
split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

print(f"Train: {train_df.index.min().date()} to {train_df.index.max().date()} ({len(train_df)} rows)")
print(f"Test:  {test_df.index.min().date()} to {test_df.index.max().date()} ({len(test_df)} rows)")

Train: 1955-01-16 to 1974-07-13 (7119 rows)
Test:  1974-07-14 to 1979-05-28 (1780 rows)


## Step 8 - save engineered features

This becomes the starting point for 03_modeling.ipynb, so we don't have to redo steps 1-7 there.

In [9]:
model_df.to_csv("../data/processed/mersey_features.csv")
print(f"Saved {len(model_df)} rows x {len(model_df.columns)} columns to data/processed/mersey_features.csv")

Saved 8899 rows x 17 columns to data/processed/mersey_features.csv


## Next steps

- [ ] Quick sanity check: plot a lag feature against actual discharge to confirm no leakage/misalignment
- [ ] Move to `03_modeling.ipynb`: XGBoost vs. a naive baseline (predict tomorrow = today), same
      MAE/RMSE framing used in Load Forecasting
- [ ] Commit this notebook + data/processed/mersey_features.csv to GitHub before starting modeling